In [ ]:
# ============================================================
# PHASE 3 - ADVERSARIAL DEFENSE EVALUATION
# ============================================================
 
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import logging
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from scipy import stats
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import (
    FastGradientMethod,
    BasicIterativeMethod,
    ProjectedGradientDescent,
    CarliniL2Method
)
 
logging.getLogger('art').setLevel(logging.ERROR)
 
# Save path — all output files saved in current directory
SAVE_PATH = ""  

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD ARTIFACTS
# ─────────────────────────────────────────────────────────────
print("Loading artifacts...")
 
le      = joblib.load('label_encoder.pkl')
X_test  = np.load('X_test.npy')
y_test  = np.load('y_test.npy')
 
# Load Phase 2 C&W artifacts
X_cw_subset = np.load('X_cw_subset.npy')   
y_cw_subset = np.load('y_cw_subset.npy')   
X_adv_cw    = np.load('X_adv_cw.npy')
 
X_subset = X_cw_subset
y_subset = y_cw_subset
 
input_dim   = X_subset.shape[1]
num_classes = len(np.unique(y_test))
 
print(f"Subset shape: {X_subset.shape}")
print(f"Classes: {le.classes_}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# SPLIT SUBSET INTO TRAIN / VAL / TEST
# ─────────────────────────────────────────────────────────────
X_temp, X_test_held, y_temp, y_test_held = train_test_split(
    X_subset, y_subset,
    test_size=0.25,
    random_state=42,
    stratify=y_subset
)
 
X_tr, X_v, y_tr, y_v = train_test_split(
    X_temp, y_temp,
    test_size=0.167,
    random_state=42,
    stratify=y_temp
)
 
y_tr_onehot = tf.keras.utils.to_categorical(y_tr, num_classes)
y_v_onehot  = tf.keras.utils.to_categorical(y_v, num_classes)
 
print(f"Train:  {X_tr.shape}")
print(f"Val:    {X_v.shape}")
print(f"Test:   {X_test_held.shape}")
print(f"\nTest set class distribution:")
for class_idx in np.unique(y_test_held):
    count = np.sum(y_test_held == class_idx)
    print(f"  {le.classes_[class_idx]}: {count}")
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# MODEL BUILDER — matches Phase 1 architecture (256-128-64-32)
# ─────────────────────────────────────────────────────────────
def build_model(input_dim, num_classes, seed=0):
    tf.random.set_seed(seed)
    np.random.seed(seed)
 
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
 
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────
# LAYER 1 — ADVERSARIAL TRAINING
# ─────────────────────────────────────────────────────────────
curriculum = [
    (0.01, 10),
    (0.05, 10),
    (0.10, 10),
]
 
def adversarial_train(model, seed, input_dim, num_classes):
    print(f"\nAdversarially training Model with seed={seed}")
 
    loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
 
    for stage, (eps, epochs) in enumerate(curriculum, 1):
        print(f"  Stage {stage}/3 — ε={eps}...")
 
        classifier = TensorFlowV2Classifier(
            model=model,
            nb_classes=num_classes,
            input_shape=(input_dim,),
            loss_object=loss_fn,
            clip_values=(float(X_tr.min()), float(X_tr.max()))
        )
 
        fgsm = FastGradientMethod(estimator=classifier, eps=eps)
        pgd  = ProjectedGradientDescent(
            estimator=classifier,
            eps=eps, eps_step=eps/10,
            max_iter=10, num_random_init=3
        )
 
        X_adv_fgsm = fgsm.generate(x=X_tr, batch_size=512)
        X_adv_pgd  = pgd.generate(x=X_tr, batch_size=256)
 
        X_combined = np.concatenate([X_tr, X_adv_fgsm, X_adv_pgd])
        y_combined = np.concatenate([y_tr_onehot, y_tr_onehot, y_tr_onehot])
 
        idx = np.random.permutation(len(X_combined))
        X_combined = X_combined[idx]
        y_combined = y_combined[idx]
 
        model.fit(
            X_combined, y_combined,
            epochs=epochs,
            batch_size=256,
            validation_data=(X_v, y_v_onehot),
            verbose=0
        )
 
        # Rewrap after each stage since weights changed
        classifier = TensorFlowV2Classifier(
            model=model,
            nb_classes=num_classes,
            input_shape=(input_dim,),
            loss_object=loss_fn,
            clip_values=(float(X_tr.min()), float(X_tr.max()))
        )
 
        y_pred = model.predict(X_test_held, verbose=0).argmax(axis=1)
        acc    = accuracy_score(y_test_held, y_pred)
        print(f"    Test accuracy after stage {stage}: {acc:.4f}")
 
    return model
 
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# VERIFY BASELINE — load full model directly
# ─────────────────────────────────────────────────────────────
print("\nVerifying baseline model...")
baseline_model = tf.keras.models.load_model('baseline_model.keras')  
y_pred_baseline = baseline_model.predict(X_test_held, verbose=0).argmax(axis=1)
baseline_acc    = accuracy_score(y_test_held, y_pred_baseline)
print(f"Baseline accuracy on test set: {baseline_acc:.4f}")
 
 
# ─────────────────────────────────────────────────────────────
# TRAIN ENSEMBLE
# ─────────────────────────────────────────────────────────────
seeds = [0, 42, 123]
ensemble_models = []
 
for seed in seeds:
    model_i = build_model(input_dim, num_classes, seed=seed)
    # Load weights from baseline into each ensemble model
    model_i.set_weights(baseline_model.get_weights())
    model_i = adversarial_train(model_i, seed, input_dim, num_classes)
    ensemble_models.append(model_i)
    model_i.save_weights(f'{SAVE_PATH}adv_model_seed{seed}.weights.h5')
    print(f"Model (seed={seed}) saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# LAYER 2 — INPUT TRANSFORMATIONS
# ─────────────────────────────────────────────────────────────
def apply_input_transformations(X, squeeze_bits=12, gaussian_std=0.01):
    X_out    = X.copy()
    data_min = X_out.min()
    data_max = X_out.max()
    scale    = (2 ** squeeze_bits) - 1
 
    X_out = (
        np.round(
            (X_out - data_min) / (data_max - data_min) * scale
        ) / scale * (data_max - data_min) + data_min
    )
 
    noise = np.random.normal(0, gaussian_std, X_out.shape)
    X_out = np.clip(X_out + noise, data_min, data_max)
 
    return X_out

In [ ]:
# ─────────────────────────────────────────────────────────────
# LAYER 3 — ENSEMBLE MAJORITY VOTING
# ─────────────────────────────────────────────────────────────
def ensemble_predict(models, X):
    all_preds = np.array([
        m.predict(X, verbose=0).argmax(axis=1)
        for m in models
    ])
    voted, _ = stats.mode(all_preds, axis=0, keepdims=True)
    return voted.flatten()
 
 
# ─────────────────────────────────────────────────────────────
# VERIFY ENSEMBLE ON CLEAN DATA
# ─────────────────────────────────────────────────────────────
print("\nEvaluating ensemble on clean test data...")
X_transformed_test  = apply_input_transformations(X_test_held)
y_pred_ensemble     = ensemble_predict(ensemble_models, X_transformed_test)
ensemble_clean_acc  = accuracy_score(y_test_held, y_pred_ensemble)
print(f"Ensemble clean accuracy: {ensemble_clean_acc:.4f}")
print(f"Baseline was:            {baseline_acc:.4f}")
print("\nPer-class performance:")
print(classification_report(
    y_test_held, y_pred_ensemble,
    target_names=le.classes_, zero_division=0
))

In [ ]:
# ─────────────────────────────────────────────────────────────
# LAYER 4 — ADVERSARIAL DETECTOR
# ─────────────────────────────────────────────────────────────
loss_fn    = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
classifier = TensorFlowV2Classifier(
    model=ensemble_models[0],
    nb_classes=num_classes,
    input_shape=(input_dim,),
    loss_object=loss_fn,
    clip_values=(float(X_tr.min()), float(X_tr.max()))
)
 
print("Generating adversarial examples for detector training...")
fgsm_det = FastGradientMethod(estimator=classifier, eps=0.05)
pgd_det  = ProjectedGradientDescent(
    estimator=classifier,
    eps=0.05, eps_step=0.005,
    max_iter=10, num_random_init=3
)
 
X_adv_det_fgsm = fgsm_det.generate(x=X_tr, batch_size=512)
X_adv_det_pgd  = pgd_det.generate(x=X_tr, batch_size=256)
 
X_detector = np.concatenate([X_tr, X_adv_det_fgsm, X_adv_det_pgd])
y_detector  = np.concatenate([
    np.zeros(len(X_tr)),
    np.ones(len(X_adv_det_fgsm)),
    np.ones(len(X_adv_det_pgd))
])
 
idx        = np.random.permutation(len(X_detector))
X_detector = X_detector[idx]
y_detector = y_detector[idx]
 
print(f"Detector training set: {X_detector.shape}")
print(f"Clean:       {int(np.sum(y_detector==0))}")
print(f"Adversarial: {int(np.sum(y_detector==1))}")
 
print("\nTraining adversarial detector...")
detector = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
detector.fit(X_detector, y_detector)
joblib.dump(detector, f'{SAVE_PATH}adversarial_detector.pkl')
print("Detector saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMBINING ALL LAYERS
# ─────────────────────────────────────────────────────────────
def defend_and_predict(X, detector, models):
    det_preds  = detector.predict(X)
    clean_mask = det_preds == 0
    adv_mask   = det_preds == 1
 
    final_preds = np.full(len(X), -1)
 
    if np.sum(clean_mask) > 0:
        X_clean_transformed    = apply_input_transformations(X[clean_mask])
        final_preds[clean_mask] = ensemble_predict(models, X_clean_transformed)
 
    return final_preds, adv_mask
 
 
def evaluate_attack(y_true, final_preds, clean_correct,
                    attack_name, eps=None, full_pipeline=False):
    classified_mask = final_preds != -1
    flagged_count   = int(np.sum(final_preds == -1))
 
    acc = (
        accuracy_score(y_true[classified_mask], final_preds[classified_mask])
        if np.sum(classified_mask) > 0 else 0.0
    )
 
    attack_success = (
        np.sum(
            clean_correct &
            (final_preds != y_true) &
            (final_preds != -1)
        ) / np.sum(clean_correct)
    )
 
    eps_label = f"e={eps}" if eps is not None else "N/A"
    print(f"\n  [{attack_name} {eps_label}]")
    print(f"  Classified:     {int(np.sum(classified_mask))}/{len(y_true)}")
    if full_pipeline:
        print(f"  Flagged:        {flagged_count}/{len(y_true)} → human review")
    print(f"  Accuracy:       {acc:.4f}")
    print(f"  Attack success: {attack_success:.4f}")
 
    return acc, attack_success, flagged_count
 
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# EVALUATION A — LAYERS 1+2+3 (no detector)
# ─────────────────────────────────────────────────────────────
epsilons  = [0.01, 0.05, 0.1, 0.15, 0.2]
results_a = []
 
classifier = TensorFlowV2Classifier(
    model=ensemble_models[0],
    nb_classes=num_classes,
    input_shape=(input_dim,),
    loss_object=loss_fn,
    clip_values=(float(X_test_held.min()), float(X_test_held.max()))
)
 
y_pred_clean  = ensemble_predict(
    ensemble_models, apply_input_transformations(X_test_held)
)
clean_correct = y_pred_clean == y_test_held
 
 
def evaluate_layers123(y_true, X_adv, clean_correct,
                       attack_name, eps=None):
    X_transformed = apply_input_transformations(X_adv)
    y_pred_adv    = ensemble_predict(ensemble_models, X_transformed)
 
    acc = accuracy_score(y_true, y_pred_adv)
    attack_success = (
        np.sum(clean_correct & (y_pred_adv != y_true))
        / np.sum(clean_correct)
    )
 
    eps_label = f"e={eps}" if eps is not None else "N/A"
    print(f"\n  [{attack_name} {eps_label}]")
    print(f"  Accuracy:       {acc:.4f}")
    print(f"  Attack success: {attack_success:.4f}")
 
    return acc, attack_success
 
 
# FGSM
print("\nFGSM vs Layers 1+2+3:")
for eps in epsilons:
    fgsm  = FastGradientMethod(estimator=classifier, eps=eps)
    X_adv = fgsm.generate(x=X_test_held, batch_size=512)
    acc, asr = evaluate_layers123(y_test_held, X_adv, clean_correct, "FGSM", eps)
    results_a.append({'Attack': 'FGSM', 'Epsilon': eps, 'Accuracy': acc, 'Attack_Success': asr})
 
# BIM
print("\nBIM vs Layers 1+2+3:")
for eps in epsilons:
    bim   = BasicIterativeMethod(estimator=classifier, eps=eps, eps_step=eps/10, max_iter=10)
    X_adv = bim.generate(x=X_test_held, batch_size=512)
    acc, asr = evaluate_layers123(y_test_held, X_adv, clean_correct, "BIM", eps)
    results_a.append({'Attack': 'BIM', 'Epsilon': eps, 'Accuracy': acc, 'Attack_Success': asr})
 
# PGD
print("\nPGD vs Layers 1+2+3:")
for eps in epsilons:
    pgd   = ProjectedGradientDescent(estimator=classifier, eps=eps, eps_step=eps/10,
                                      max_iter=10, num_random_init=5)
    X_adv = pgd.generate(x=X_test_held, batch_size=256)
    acc, asr = evaluate_layers123(y_test_held, X_adv, clean_correct, "PGD", eps)
    results_a.append({'Attack': 'PGD', 'Epsilon': eps, 'Accuracy': acc, 'Attack_Success': asr})
 
# C&W — reuse adversarial examples from Phase 2
print("\nC&W vs Layers 1+2+3:")
CW_SAMPLES = 100
cw_indices = []
for class_idx in np.unique(y_test_held):
    class_mask = np.where(y_test_held == class_idx)[0]
    chosen     = np.random.choice(class_mask, size=min(CW_SAMPLES, len(class_mask)), replace=False)
    cw_indices.extend(chosen)
 
X_cw = X_test_held[cw_indices]
y_cw = y_test_held[cw_indices]
 
clean_correct_cw = (
    ensemble_predict(ensemble_models, apply_input_transformations(X_cw)) == y_cw
)
 
classifier_cw = TensorFlowV2Classifier(
    model=ensemble_models[0],
    nb_classes=num_classes,
    input_shape=(input_dim,),
    loss_object=loss_fn,
    clip_values=(float(X_cw.min()), float(X_cw.max()))
)
cw_attack = CarliniL2Method(
    classifier=classifier_cw,
    confidence=0.0,
    max_iter=50,
    learning_rate=0.01,
    binary_search_steps=3,
    batch_size=32
)
X_adv_cw_new = cw_attack.generate(x=X_cw)
acc, asr     = evaluate_layers123(y_cw, X_adv_cw_new, clean_correct_cw, "C&W")
results_a.append({'Attack': 'C&W', 'Epsilon': 'N/A', 'Accuracy': acc, 'Attack_Success': asr})
 
df_a = pd.DataFrame(results_a)
print("\nEvaluation A Summary:")
print(df_a.to_string(index=False))
df_a.to_csv(f'{SAVE_PATH}eval_a_layers123.csv', index=False)

In [ ]:
# ─────────────────────────────────────────────────────────────
# EVALUATION C — FULL PIPELINE (all 4 layers)
# ─────────────────────────────────────────────────────────────
results_c = []
 
clean_full_preds, clean_flagged = defend_and_predict(X_test_held, detector, ensemble_models)
classified_mask = clean_full_preds != -1
clean_full_acc  = accuracy_score(
    y_test_held[classified_mask], clean_full_preds[classified_mask]
)
print(f"\nClean data through full system:")
print(f"  Classified: {int(np.sum(classified_mask))}/{len(X_test_held)}")
print(f"  Flagged:    {int(np.sum(clean_flagged))}/{len(X_test_held)}")
print(f"  Accuracy:   {clean_full_acc:.4f}")
 
clean_correct_full = (
    (clean_full_preds == y_test_held) | (clean_full_preds == -1)
)
 
# BIM through full pipeline
for eps in epsilons:
    bim   = BasicIterativeMethod(estimator=classifier, eps=eps, eps_step=eps/10, max_iter=10)
    X_adv = bim.generate(x=X_test_held, batch_size=512)
    final_preds, flagged = defend_and_predict(X_adv, detector, ensemble_models)
    acc, asr, flagged_count = evaluate_attack(
        y_test_held, final_preds, clean_correct_full, "BIM", eps, full_pipeline=True
    )
    results_c.append({
        'Attack': 'BIM', 'Epsilon': eps,
        'Accuracy': acc, 'Attack_Success': asr, 'Flagged': flagged_count
    })
 
# C&W through full pipeline
clean_preds_cw, _ = defend_and_predict(X_cw, detector, ensemble_models)
clean_correct_cw_full = (clean_preds_cw == y_cw) | (clean_preds_cw == -1)
 
final_preds_cw, flagged_cw = defend_and_predict(X_adv_cw_new, detector, ensemble_models)
acc_cw, asr_cw, flagged_cw_count = evaluate_attack(
    y_cw, final_preds_cw, clean_correct_cw_full, "C&W", full_pipeline=True
)
results_c.append({
    'Attack': 'C&W', 'Epsilon': 'N/A',
    'Accuracy': acc_cw, 'Attack_Success': asr_cw, 'Flagged': flagged_cw_count
})
 
df_c = pd.DataFrame(results_c)
print("\nEvaluation C Summary:")
print(df_c.to_string(index=False))
df_c.to_csv(f'{SAVE_PATH}eval_c_full_pipeline.csv', index=False)
 
print("\nPhase 3 complete!")
print("Files saved:")
print(f"  {SAVE_PATH}adv_model_seed0.weights.h5")
print(f"  {SAVE_PATH}adv_model_seed42.weights.h5")
print(f"  {SAVE_PATH}adv_model_seed123.weights.h5")
print(f"  {SAVE_PATH}adversarial_detector.pkl")
print(f"  {SAVE_PATH}eval_a_layers123.csv")
print(f"  {SAVE_PATH}eval_c_full_pipeline.csv")